# SQL Query Documentation & Optimization

## Customer Spending Behavior & Revenue Analytics

### Purpose

This notebook documents the SQL queries developed for the Online Retail II project.

The relational database was implemented in **MySQL** and contains four main tables:

- `customers`
- `invoices`
- `invoice_items`
- `products`

The SQL queries answer business questions related to customer value, geographic revenue, product performance, cancellation behavior, customer ranking, and monthly cancellation trends.

The main SQL queries are stored in:

`sql/business_queries.sql`

This notebook documents the joins, filters, conditions, aggregations, and optimization techniques used in these queries.

## 1. Database Relationships

The database contains four main tables:

- `customers` — customer ID and country
- `invoices` — invoice information, date, customer ID and cancellation status
- `invoice_items` — quantity, price and revenue for each invoice item
- `products` — product description and category

The main relationships are:

- `customers.customer_id` → `invoices.customer_id`
- `invoices.invoice_id` → `invoice_items.invoice_id`
- `products.stock_code` → `invoice_items.stock_code`

These relationships allow customer, transaction and product information to be combined using SQL joins.

## 2. SQL Business Queries

### Query 1 — Top 10 Customers by Revenue

**Objective:** Identify the customers generating the highest completed revenue.

**Joins:**  
`customers` is joined to `invoices` using `customer_id`, and `invoices` is joined to `invoice_items` using `invoice_id`.

**Filter:**  
`WHERE i.is_cancelled = 0` excludes cancelled invoices.

**Aggregation:**  
`SUM(ii.revenue)` calculates completed revenue for each customer.

**Sorting:**  
Customers are ordered by total revenue in descending order and `LIMIT 10` returns the ten highest-value customers.

```sql
SELECT
    c.customer_id,
    c.country,
    SUM(ii.revenue) AS total_revenue
FROM customers c
JOIN invoices i
    ON c.customer_id = i.customer_id
JOIN invoice_items ii
    ON i.invoice_id = ii.invoice_id
WHERE i.is_cancelled = 0
GROUP BY c.customer_id, c.country
ORDER BY total_revenue DESC
LIMIT 10;
```

### Query 2 — Revenue by Country

**Objective:** Compare completed revenue across geographic markets.

**Joins:**  
Customer country information is connected to invoices using `customer_id`, and invoice revenue is connected using `invoice_id`.

**Filter:**  
Cancelled invoices are excluded so that the analysis represents completed revenue.

**Aggregation:**  
`SUM(ii.revenue)` calculates total revenue by country, while `COUNT(DISTINCT i.invoice_id)` calculates completed orders.

**Grouping:**  
Results are grouped by country and sorted from highest to lowest revenue.

### Query 3 — Revenue by Product Category

**Objective:** Identify the product categories generating the highest completed revenue.

**Joins:**  
`invoice_items` is joined to `invoices` using `invoice_id` and to `products` using `stock_code`.

**Filter:**  
Cancelled invoices are excluded.

**Aggregation:**  
`SUM(ii.revenue)` calculates revenue by category and `SUM(ii.quantity)` calculates the total quantity sold.

**Grouping:**  
Results are grouped by product category and sorted by revenue in descending order.

### Query 4 — High-Value Customers with Cancellation Activity

**Objective:** Identify high-value customers who also demonstrate cancellation activity.

**Conditional aggregation:**  
`CASE WHEN` is used to separate completed revenue from cancelled orders.

**Condition:**  
Customers are retained when completed revenue is greater than £1,000 and their cancellation rate is greater than 0%.

**Filtering after aggregation:**  
`HAVING` is used because these conditions depend on aggregated values.

This query helps identify valuable customers whose cancellation behavior may represent potential revenue risk.

### Query 5 — Customer Revenue Ranking by Country

**Objective:** Rank customers according to completed revenue within each country.

**Window function:**  
`RANK()` ranks customers according to their completed revenue.

**Partitioning:**  
`PARTITION BY c.country` creates a separate ranking for each country.

This allows high value customers to be identified within individual geographic markets rather than only at the global level.

### Query 6 — Customers Above Average Revenue

**Objective:** Identify customers whose completed revenue is higher than the average completed revenue per customer.

**Subquery:**  
A subquery first calculates total completed revenue for each customer.

The average of these customer level revenues is then calculated.

The outer query compares each customer's revenue with this average.

This allows customers performing above the overall customer benchmark to be identified.

### Query 7 — Monthly Cancellation Rate

**Objective:** Analyze how the cancellation rate changes over time.

**Date transformation:**  
`DATE_FORMAT(invoice_date, '%Y-%m')` converts invoice dates into monthly periods.

**Conditional aggregation:**  
`CASE WHEN is_cancelled = 1` identifies cancelled invoices.

**Calculation:**  
Cancelled invoices are divided by total invoices and multiplied by 100 to calculate the monthly cancellation rate.

**Grouping:**  
Invoices are grouped by month to compare cancellation behavior over time.

## 3. SQL Query Optimization

The database uses primary keys and indexes on the main join columns:

- `customers.customer_id` — PRIMARY KEY
- `invoices.invoice_id` — PRIMARY KEY
- `invoices.customer_id` — indexed
- `invoice_items.invoice_id` — indexed
- `invoice_items.stock_code` — indexed
- `products.stock_code` — PRIMARY KEY

The analytical queries select only the required columns rather than using `SELECT *`.

Where appropriate, cancelled invoices are filtered before aggregation.

`LIMIT` is used when only a specific number of ranked results is required.

MySQL `EXPLAIN` was used to inspect how the database executes the Top 10 Customers query.

## 4. Query Execution Plan

MySQL `EXPLAIN` was used to analyze the execution plan of the Top 10 Customers query.

```sql
EXPLAIN
SELECT
    c.customer_id,
    c.country,
    SUM(ii.revenue) AS total_revenue
FROM customers c
JOIN invoices i
    ON c.customer_id = i.customer_id
JOIN invoice_items ii
    ON i.invoice_id = ii.invoice_id
WHERE i.is_cancelled = 0
GROUP BY c.customer_id, c.country
ORDER BY total_revenue DESC
LIMIT 10;
```

The execution plan showed that:

- `customers` uses its `PRIMARY` key for the join
- `invoice_items` uses the `invoice_id` index
- `invoices` performs a table scan

The `is_cancelled` column has low cardinality because it contains a limited number of values. Therefore, adding a standalone index to this field would not necessarily improve query performance.

The execution plan was used to verify the existing indexing strategy rather than adding unnecessary indexes.

## 5. Conclusion

The SQL component of the project demonstrates data extraction and analysis from a relational MySQL database using:

- multi-table joins
- filtering and conditions
- aggregation
- conditional aggregation
- `HAVING`
- subqueries
- window functions
- date transformations
- sorting and result limitation

Query performance was reviewed using primary and foreign key indexes and MySQL `EXPLAIN`.

The SQL queries are stored in `sql/business_queries.sql` and version-controlled with Git.